In [12]:
#Imports, load and prepare data
import numpy as np
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error

from data_loader import load_data
from feature_engineering import add_engineered_features

#Database path
db_path = r"C:\Users\jacov\anaconda_projects\Marvel Insurance\data\superhero_events_db.duckdb"
df = load_data(db_path)
df = add_engineered_features(df)

print("Initial dataset shape:", df.shape)

#Drop corrupted XML row - Kratos. This will be automated in the future.
df = df.dropna(subset=["credit_score", "age"])
print("After dropping Kratos XML:", df.shape)

#Keep only rows with known target 
df = df.dropna(subset=["annual_public_destruction_events"])
print("After dropping missing targets:", df.shape) #it should be 69

Initial dataset shape: (91, 10)
After dropping Kratos XML: (90, 10)
After dropping missing targets: (69, 10)


In [14]:
#sanity check
print("\nRemaining missing values:")
print(df.isnull().sum()) 


Remaining missing values:
index                               0
name                                0
CreditInfo                          0
annual_public_destruction_events    0
credit_score                        0
age                                 0
num_superpowers                     0
num_properties                      0
num_credit_cards                    0
total_credit_limit                  0
dtype: int64


In [16]:
#Linear Regression
#Feature selection
features = ["credit_score", "num_properties", "age"] #Top features selected with highest correlation

X = df[features]
y = df["annual_public_destruction_events"]

#Model choice
model = LinearRegression()

#10-Fold Cross Validation
kf = KFold(n_splits=10, shuffle=True, random_state=42)

#Calc MAE
mae_scores = -cross_val_score(model, X, y, cv=kf, scoring="neg_mean_absolute_error")

#Calc RMSE
rmse_scores = np.sqrt(-cross_val_score(model, X, y, cv=kf, scoring="neg_mean_squared_error"))

print("Linear Regression Cross-Validaiton Results: ")
for i, (mae, rmse) in enumerate(zip(mae_scores, rmse_scores), 1):
    print(f" Fold {i}:  MAE = {mae:.3f} | RMSE = {rmse:.3f}")

print(" ")
print(f"Average MAE:  {mae_scores.mean():.3f}")
print(f"Average RMSE: {rmse_scores.mean():.3f}")

#Train final model
model.fit(X, y)

joblib.dump(model, "Marvelous_prediction_model_lr.joblib")
print("\nMarvelous_prediction_model_lr saved.")

Linear Regression Cross-Validaiton Results: 
 Fold 1:  MAE = 2.493 | RMSE = 3.006
 Fold 2:  MAE = 5.876 | RMSE = 6.713
 Fold 3:  MAE = 4.994 | RMSE = 5.379
 Fold 4:  MAE = 4.768 | RMSE = 5.311
 Fold 5:  MAE = 3.007 | RMSE = 3.642
 Fold 6:  MAE = 4.241 | RMSE = 5.030
 Fold 7:  MAE = 4.717 | RMSE = 5.745
 Fold 8:  MAE = 3.156 | RMSE = 4.854
 Fold 9:  MAE = 6.191 | RMSE = 9.342
 Fold 10:  MAE = 2.487 | RMSE = 3.922
 
Average MAE:  4.193
Average RMSE: 5.294

Marvelous_prediction_model_lr saved.


In [18]:
#Random Forest

from sklearn.ensemble import RandomForestRegressor

#Model choice
rf_model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)

#10-Fold Cross Validation
rf_mae_scores = -cross_val_score(rf_model, X, y, cv=kf, scoring="neg_mean_absolute_error")
rf_rmse_scores = np.sqrt(-cross_val_score(rf_model, X, y, cv=kf, scoring="neg_mean_squared_error"))

print("\nRandom Forest Cross-Validation Results:")
for i, (mae, rmse) in enumerate(zip(rf_mae_scores, rf_rmse_scores), 1):
    print(f" Fold {i}:  MAE = {mae:.3f} | RMSE = {rmse:.3f}")

print(" ")
print(f"Average MAE:  {rf_mae_scores.mean():.3f}")
print(f"Average RMSE: {rf_rmse_scores.mean():.3f}")

rf_model.fit(X, y)
joblib.dump(rf_model, "Marvelous_prediction_model_rf.joblib")
print("\nMarvelous_prediction_model_rf saved.")


Random Forest Cross-Validation Results:
 Fold 1:  MAE = 1.660 | RMSE = 2.061
 Fold 2:  MAE = 5.379 | RMSE = 6.207
 Fold 3:  MAE = 4.046 | RMSE = 5.025
 Fold 4:  MAE = 4.979 | RMSE = 6.334
 Fold 5:  MAE = 3.114 | RMSE = 3.771
 Fold 6:  MAE = 3.213 | RMSE = 3.668
 Fold 7:  MAE = 2.826 | RMSE = 4.379
 Fold 8:  MAE = 3.381 | RMSE = 4.084
 Fold 9:  MAE = 5.301 | RMSE = 7.678
 Fold 10:  MAE = 2.645 | RMSE = 3.028
 
Average MAE:  3.654
Average RMSE: 4.624

Marvelous_prediction_model_rf saved.


In [20]:
#Final Model: Random Forest
final_model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)

#Train on full cleaned dataset
final_model.fit(X, y)

#Save production model
joblib.dump(final_model, "Marvelous_prediction_model.joblib")
print("\nFinal model (Random Forest) saved as 'Marvelous_prediction_model.joblib'")


Final model (Random Forest) saved as 'Marvelous_prediction_model.joblib'
